# Занятие 10. Лабораторная: полносвязная сеть на CIFAR-10

Срок сдачи — 14 ноября 2026 года, ноутбук присылают преподавателю.
В лабораторной семь задач, вместе они дают до 10 баллов; задачи 3, 4 и 5 весят вдвое
больше остальных. Итог округляется вниз.

Цель — собрать из модулей семинара сеть заданной архитектуры и решить ей классификацию
цветных картинок CIFAR-10: подготовить данные, обучить сеть с возвратом к лучшей эпохе,
разобрать ошибки по классам и сравнить сеть с моделями sklearn.

**Заготовки.** В каждой заготовке указано, что должно получиться: тип
и форма в аннотации, размеры входов и выходов в комментарии перед функцией.
Имена переменных, функций, классов и атрибутов менять нельзя, проверка ищет
их по именам. Вспомогательные функции можно добавлять свои; код семинара
можно переносить. После каждой задачи идет ячейка с открытыми проверками;
при сдаче работа дополнительно проверяется закрытыми тестами на других данных.

**Что можно использовать.** numpy для своей сети и ее обучения. PyTorch, TensorFlow,
JAX и автоматическое дифференцирование нельзя. Модели sklearn нужны только в задаче 7
для сравнения.

In [ ]:
import pickle
import tarfile
import time
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import get_data_home
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
SEED = 42

## Данные: CIFAR-10

CIFAR-10 (Canadian Institute For Advanced Research) — 60 000 цветных картинок 32×32 пикселя
десяти классов, по 6000 на класс: 50 000 для обучения и 10 000 в тестовом сете. Картинка
хранится строкой из 3072 чисел от 0 до 255: сначала 1024 значения красного канала,
затем 1024 зеленого и 1024 синего; внутри канала пиксели идут по строкам.

Архив занимает 163 МБ. Код ниже скачивает его один раз в папку данных sklearn
(та же, куда `fetch_openml` кладет MNIST) и при следующих запусках читает оттуда.

In [ ]:
CIFAR_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
CIFAR_CLASSES = ["самолет", "автомобиль", "птица", "кошка", "олень",
                 "собака", "лягушка", "лошадь", "корабль", "грузовик"]


def load_cifar10():
    """Обучающие и тестовые картинки CIFAR-10: (50000, 3072) и (10000, 3072) uint8, метки 0–9."""
    data_home = Path(get_data_home())
    folder = data_home / "cifar-10-batches-py"
    if not folder.exists():
        archive = data_home / "cifar-10-python.tar.gz"
        if not archive.exists():
            print("скачиваем CIFAR-10, 163 МБ")
            urllib.request.urlretrieve(CIFAR_URL, archive)
        with tarfile.open(archive) as opened:
            opened.extractall(data_home)

    parts = []
    for name in [f"data_batch_{k}" for k in range(1, 6)] + ["test_batch"]:
        with open(folder / name, "rb") as file:
            batch = pickle.load(file, encoding="bytes")
        parts.append((batch[b"data"], np.array(batch[b"labels"])))

    X_learn = np.concatenate([X for X, _ in parts[:5]])
    y_learn = np.concatenate([y for _, y in parts[:5]])
    X_holdout, y_holdout = parts[5]
    return X_learn, y_learn, X_holdout, y_holdout


cifar_X, cifar_y, cifar_X_test, cifar_y_test = load_cifar10()
print("обучающая часть:", cifar_X.shape, cifar_X.dtype, "| тестовый сет:", cifar_X_test.shape)

In [ ]:
# яркость в долях от 0 до 1; float32 вдвое экономнее float64: 45 000 картинок — 550 МБ вместо 1.1 ГБ
X_train = cifar_X[:45000].astype(np.float32) / 255
y_train = cifar_y[:45000]
X_val = cifar_X[45000:].astype(np.float32) / 255
y_val = cifar_y[45000:]
X_test = cifar_X_test.astype(np.float32) / 255
y_test = cifar_y_test

print("обучение:", X_train.shape, "валидация:", X_val.shape, "тест:", X_test.shape)
print("картинок каждого класса в обучении:", np.bincount(y_train))

In [ ]:
def to_images(X):
    """Строки (n, 3072) -> картинки (n, 32, 32, 3) для plt.imshow."""
    images = X.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return images


def show_images(X, labels, predicted=None, n_columns=10):
    """Сетка картинок; над каждой — класс, а если передано предсказание, то «класс → предсказание»."""
    images = to_images(X)
    n_rows = int(np.ceil(len(images) / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(1.5 * n_columns, 1.75 * n_rows))
    for k, ax in enumerate(np.atleast_1d(axes).ravel()):
        ax.axis("off")
        if k >= len(images):
            continue
        ax.imshow(np.clip(images[k], 0, 1))
        title = CIFAR_CLASSES[labels[k]]
        color = BLACK
        if predicted is not None:
            title = f"{title} →\n{CIFAR_CLASSES[predicted[k]]}"
            color = BLACK if predicted[k] == labels[k] else "#B5524B"
        ax.set_title(title, fontsize=8.5, color=color)
    plt.tight_layout()
    plt.show()


first_of_each_class = [int(np.flatnonzero(y_train == label)[0]) for label in range(10)]
second_of_each_class = [int(np.flatnonzero(y_train == label)[1]) for label in range(10)]
examples = first_of_each_class + second_of_each_class
show_images(X_train[examples], y_train[examples])

## Блок A. Модули и данные

### Задача 1. Модули семинара

Перенесите из семинара полносвязный слой и ReLU. Интерфейс тот же: `forward(X)` возвращает
выход и запоминает то, что нужно для обратного прохода; `backward(grad_output)` по градиенту
потери по выходу $G = \frac{\partial L}{\partial Y}$ считает градиенты по параметрам и возвращает градиент по входу;
`parameters()` возвращает список пар (параметр, градиент).

- `Linear(n_in, n_out, rng)`: $Y = X W + b$; веса $W$ формы $(n_{in}, n_{out})$ — `rng.normal` со средним 0
  и дисперсией $\frac{2}{n_{in}}$ (инициализация He), сдвиг $b$ — нули;
  $\frac{\partial L}{\partial W} = X^\top G$, $\frac{\partial L}{\partial b} = \sum_i G_{i\cdot}$, $\frac{\partial L}{\partial X} = G\, W^\top$;
  градиенты — в атрибутах `dW` и `db`;
- `ReLU`: $\max(z, 0)$ поэлементно, производная $[z > 0]$.

In [ ]:
class Linear:
    # W: (n_in, n_out) ~ N(0, 2 / n_in) через rng.normal; b: (n_out,) нули; dW, db — нули той же формы
    def __init__(self, n_in: int, n_out: int, rng: np.random.Generator):
        ...

    # X: (n, n_in) -> (n, n_out)
    def forward(self, X: np.ndarray) -> np.ndarray:
        return ...

    # grad_output: (n, n_out); записывает self.dW, self.db; возвращает (n, n_in)
    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        return ...

    # [(self.W, self.dW), (self.b, self.db)]
    def parameters(self) -> list:
        return ...


class ReLU:
    # X -> та же форма
    def forward(self, X: np.ndarray) -> np.ndarray:
        return ...

    # grad_output -> та же форма
    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        return ...

    def parameters(self) -> list:
        return []

In [ ]:
# --- проверка ---
layer = Linear(3, 2, np.random.default_rng(0))
assert layer.W.shape == (3, 2) and layer.b.shape == (2,) and np.all(layer.b == 0), "W — (n_in, n_out), b — нули (n_out,)"
X_small = np.arange(6.0).reshape(2, 3)
assert np.allclose(layer.forward(X_small), X_small @ layer.W), "forward — X W + b"
grad_input = layer.backward(np.ones((2, 2)))
assert np.allclose(layer.dW, X_small.T @ np.ones((2, 2))) and np.allclose(layer.db, [2, 2]), "dW = X^T G, db — сумма по объектам"
assert np.allclose(grad_input, np.ones((2, 2)) @ layer.W.T), "градиент по входу — G W^T"

relu = ReLU()
z = np.array([[-2.0, 0.0, 3.0]])
assert np.array_equal(relu.forward(z), [[0.0, 0.0, 3.0]]), "ReLU — max(z, 0)"
assert np.array_equal(relu.backward(np.ones_like(z)), [[0.0, 0.0, 1.0]]), "производная ReLU — [z > 0]"
print("проверки пройдены")

Остальное — как на семинаре и уже готово: софтмакс с кросс-энтропией, контейнер `Sequential`,
градиентный спуск `SGD` и выдача мини-батчей. `accuracy` считает ответы кусками по 5000 картинок:
сеть умножает float32-картинки на float64-веса, и прогон всех 45 000 картинок разом создал бы
промежуточную копию больше гигабайта.

In [ ]:
class SoftmaxCrossEntropy:
    """Средняя кросс-энтропия софтмакса оценок по объектам."""

    def forward(self, logits: np.ndarray, y: np.ndarray) -> float:
        shifted = logits - logits.max(axis=1, keepdims=True)        # устойчивость: exp не переполняется
        exp_scores = np.exp(shifted)
        self.probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)
        self.y = y

        n = len(y)
        true_class_probability = self.probabilities[np.arange(n), y]
        loss = float(-np.mean(np.log(true_class_probability + 1e-12)))
        return loss

    def backward(self) -> np.ndarray:
        n = len(self.y)
        grad_logits = self.probabilities.copy()
        grad_logits[np.arange(n), self.y] -= 1.0
        grad_logits = grad_logits / n
        return grad_logits


class Sequential:
    """Слои по очереди: forward слева направо, backward справа налево."""

    def __init__(self, layers: list):
        self.layers = layers

    def forward(self, X: np.ndarray) -> np.ndarray:
        output = X
        for layer in self.layers:
            output = layer.forward(output)
        return output

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        grad = grad_output
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def parameters(self) -> list:
        pairs = [pair for layer in self.layers for pair in layer.parameters()]
        return pairs


class SGD:
    """Градиентный спуск: параметр ← параметр − lr · градиент, на месте."""

    def __init__(self, lr: float):
        self.lr = lr

    def step(self, parameters: list) -> None:
        for parameter, gradient in parameters:
            parameter -= self.lr * gradient


def iterate_minibatches(X: np.ndarray, y: np.ndarray, batch_size: int, rng: np.random.Generator):
    """Перемешать объекты и выдать их пачками по batch_size; каждый объект — ровно один раз за эпоху."""
    order = rng.permutation(len(y))
    for start in range(0, len(y), batch_size):
        batch = order[start:start + batch_size]
        yield X[batch], y[batch]


def accuracy(model, X: np.ndarray, y: np.ndarray, chunk: int = 5000) -> float:
    """Доля верных ответов; прямой проход кусками по chunk объектов."""
    correct = 0
    for start in range(0, len(y), chunk):
        logits = model.forward(X[start:start + chunk])
        correct += int(np.sum(logits.argmax(axis=1) == y[start:start + chunk]))
    value = correct / len(y)
    return value

### Задача 2. Стандартизация

Яркость пикселей от 0 до 1 и в среднем около 0.45: все входы сдвинуты в одну сторону.
Стандартизация делает у каждого из 3072 признаков нулевое среднее и единичное стандартное
отклонение: $\tilde x_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$, где $\mu_j$ и $\sigma_j$ — среднее и стандартное отклонение признака $j$.

$\mu$ и $\sigma$ считают только по обучающей выборке и теми же числами преобразуют валидацию
и тестовый сет — иначе в обучение попадут сведения о тестовых картинках.
`fit_standardizer` возвращает `(mean, std)`, `standardize` применяет их. Сохраните среднее
и стандартное отклонение обучения в `pixel_mean` и `pixel_std` и замените `X_train`, `X_val`,
`X_test` стандартизованными.

In [ ]:
# X: (n, d) -> (mean, std), обе формы (d,)
def fit_standardizer(X: np.ndarray) -> tuple:
    return ...


# X: (n, d), mean и std: (d,) -> (n, d) float32
def standardize(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ...


pixel_mean: np.ndarray = ...      # (3072,) по обучению
pixel_std: np.ndarray = ...       # (3072,) по обучению
# X_train, X_val, X_test — заменить стандартизованными

In [ ]:
# --- проверка ---
assert pixel_mean.shape == pixel_std.shape == (3072,), "среднее и стандартное отклонение — по каждому из 3072 признаков"
assert X_train.dtype == np.float32, "стандартизованные данные — float32"
assert np.allclose(X_train[:, :50].mean(axis=0), 0, atol=1e-3), "у обучения среднее каждого признака — 0"
assert np.allclose(X_train[:, :50].std(axis=0), 1, atol=1e-3), "у обучения стандартное отклонение каждого признака — 1"
assert abs(X_test.mean()) < 0.1, "тестовый сет преобразован теми же числами"
print("среднее признаков валидации:", round(float(X_val.mean()), 4), "— не ровно 0: числа взяты с обучения")
print("проверки пройдены")

## Блок B. Сеть

### Задача 3. Архитектура

Соберите сеть ровно такой архитектуры:

| № | модуль | вход → выход | параметров |
|:--|:--|:--|--:|
| 1 | `Linear` | 3072 → 512 | $3072 \cdot 512 + 512 = 1\,573\,376$ |
| 2 | `ReLU` | 512 → 512 | 0 |
| 3 | `Linear` | 512 → 256 | $512 \cdot 256 + 256 = 131\,328$ |
| 4 | `ReLU` | 256 → 256 | 0 |
| 5 | `Linear` | 256 → 10 | $256 \cdot 10 + 10 = 2570$ |

После последнего линейного слоя активации нет: его выход — десять оценок, а софтмакс
входит в потерю `SoftmaxCrossEntropy`. `build_network(rng)` возвращает `Sequential` из этих
пяти модулей; слои `Linear` создаются по порядку с переданным `rng`. `count_parameters(network)`
считает число чисел во всех параметрах сети.

In [ ]:
# Linear(3072, 512) -> ReLU -> Linear(512, 256) -> ReLU -> Linear(256, 10); Linear создаются по порядку с rng
def build_network(rng: np.random.Generator) -> Sequential:
    return ...


# число элементов во всех параметрах network
def count_parameters(network: Sequential) -> int:
    return ...

In [ ]:
# --- проверка ---
network = build_network(np.random.default_rng(0))
kinds = [type(layer).__name__ for layer in network.layers]
assert kinds == ["Linear", "ReLU", "Linear", "ReLU", "Linear"], f"порядок модулей: {kinds}"
shapes = [layer.W.shape for layer in network.layers if isinstance(layer, Linear)]
assert shapes == [(3072, 512), (512, 256), (256, 10)], f"формы весов: {shapes}"
assert network.forward(X_train[:4]).shape == (4, 10), "выход — десять оценок на картинку"
assert count_parameters(network) == 1_707_274, f"параметров {count_parameters(network)}, по таблице 1 707 274"
print("проверки пройдены")

### Задача 4. Обучение с возвратом к лучшей эпохе

Полносвязная сеть на CIFAR-10 быстро переобучается: accuracy на обучении растет, а на
валидации после нескольких эпох перестает расти и падает. Поэтому после обучения сеть
возвращают к весам той эпохи, на которой accuracy на валидации была наибольшей.

`train` проходит `n_epochs` эпох по мини-батчам `iterate_minibatches`: прямой проход,
потеря `SoftmaxCrossEntropy`, обратный проход, шаг `optimizer.step(model.parameters())`.
После каждой эпохи в историю записываются средняя потеря по пачкам и accuracy на обучении
и на валидации. Копию параметров лучшей эпохи сохраняют; после последней эпохи параметры
сети заменяют этой копией на месте — так, чтобы `model.parameters()` вернул те же массивы.
При равной accuracy лучшей считается более ранняя эпоха.

In [ ]:
# возвращает {"потеря на обучении": [...], "accuracy на обучении": [...], "accuracy на валидации": [...]},
# каждый список длины n_epochs; после выхода параметры model — как после лучшей по валидации эпохи
def train(model, optimizer, X: np.ndarray, y: np.ndarray, X_val: np.ndarray, y_val: np.ndarray,
          n_epochs: int, batch_size: int, rng: np.random.Generator) -> dict:
    return ...

In [ ]:
# --- проверка: 3 эпохи на 5000 картинках ---
check_rng = np.random.default_rng(SEED)
check_network = build_network(check_rng)
check_history = train(check_network, SGD(lr=0.01), X_train[:5000], y_train[:5000], X_val, y_val,
                      n_epochs=3, batch_size=128, rng=check_rng)
assert set(check_history) == {"потеря на обучении", "accuracy на обучении", "accuracy на валидации"}, "ключи истории"
assert all(len(values) == 3 for values in check_history.values()), "по значению на эпоху"
assert check_history["потеря на обучении"][-1] < check_history["потеря на обучении"][0], "потеря на обучении падает"
restored = accuracy(check_network, X_val, y_val)
assert np.isclose(restored, max(check_history["accuracy на валидации"])), "после train сеть — как после лучшей эпохи"
print("accuracy на валидации по эпохам:", np.round(check_history["accuracy на валидации"], 4))
print("проверки пройдены")

### Задача 5. Итоговая сеть

Обучите сеть `build_network` так, чтобы accuracy на тестовом сете `final_test_accuracy` была
не меньше 0.50, а обучение заняло не больше 5 минут (`final_train_time`, секунды).
Архитектуру менять нельзя; шаг, размер пачки и число эпох выберите сами по валидации — для сравнения: случайный ответ дает 0.10. Историю сохраните в `final_history`
и нарисуйте кривые потери и accuracy по эпохам.

In [ ]:
final_network: Sequential = ...      # build_network(...), обученная train
final_history: dict = ...            # результат train для final_network
final_train_time: float = ...        # секунды обучения
final_test_accuracy: float = ...     # accuracy(final_network, X_test, y_test)

In [ ]:
# --- проверка ---
kinds = [type(layer).__name__ for layer in final_network.layers]
assert kinds == ["Linear", "ReLU", "Linear", "ReLU", "Linear"], "архитектура — из задачи 3"
assert set(final_history) == {"потеря на обучении", "accuracy на обучении", "accuracy на валидации"}, "ключи истории"
assert np.isclose(final_test_accuracy, accuracy(final_network, X_test, y_test)), "final_test_accuracy — accuracy на X_test"
assert final_test_accuracy >= 0.5, f"accuracy на тесте {final_test_accuracy:.4f}, нужно не меньше 0.5"
assert final_train_time <= 300, f"обучение заняло {final_train_time:.0f} с, нужно не больше 300"
print(f"accuracy на тесте {final_test_accuracy:.4f}, обучение {final_train_time:.0f} с")
print("проверки пройдены")

## Блок C. Разбор и сравнение

### Задача 6. Ошибки по классам

По предсказаниям `final_network` на тестовом сете посчитайте:

- `confusion` — матрицу ошибок: массив $10 \times 10$, в строке $i$ и столбце $j$ — число картинок
  класса $i$, которым сеть дала ответ $j$;
- `class_accuracy` — `pd.Series` с индексом `CIFAR_CLASSES`: доля верных ответов в каждом классе;
- `most_confused` — пару названий `(верный класс, ответ сети)` для самой частой ошибки,
  то есть наибольшего элемента `confusion` вне диагонали.

In [ ]:
confusion: np.ndarray = ...          # (10, 10) int: строка — верный класс, столбец — ответ сети
class_accuracy: pd.Series = ...      # индекс CIFAR_CLASSES, доля верных ответов в классе
most_confused: tuple = ...           # (название верного класса, название ответа сети)

In [ ]:
# --- проверка ---
assert confusion.shape == (10, 10) and confusion.sum() == len(y_test), "матрица 10 × 10, сумма — число тестовых картинок"
assert np.array_equal(confusion.sum(axis=1), np.bincount(y_test, minlength=10)), "строка — верный класс"
assert list(class_accuracy.index) == CIFAR_CLASSES, "индекс — CIFAR_CLASSES"
assert np.isclose((class_accuracy * np.bincount(y_test)).sum() / len(y_test), final_test_accuracy), "средневзвешенная accuracy классов — общая"
assert len(most_confused) == 2 and most_confused[0] != most_confused[1], "пара разных классов"

fig, (ax_matrix, ax_bars) = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.2, 1]})
image = ax_matrix.imshow(confusion, cmap="Blues")
ax_matrix.set_xticks(range(10))
ax_matrix.set_xticklabels(CIFAR_CLASSES, rotation=60, ha="right", fontsize=9)
ax_matrix.set_yticks(range(10))
ax_matrix.set_yticklabels(CIFAR_CLASSES, fontsize=9)
ax_matrix.set_xlabel("ответ сети")
ax_matrix.set_ylabel("верный класс")
ax_matrix.grid(False)
for i in range(10):
    for j in range(10):
        ax_matrix.text(j, i, confusion[i, j], ha="center", va="center", fontsize=7.5,
                       color="white" if confusion[i, j] > confusion.max() / 2 else BLACK)
order = class_accuracy.sort_values()
ax_bars.barh(order.index, order.values, color=BLUE)
ax_bars.axvline(final_test_accuracy, color=OCHRE, ls="--", label=f"вся выборка: {final_test_accuracy:.3f}")
ax_bars.set_xlabel("accuracy в классе")
ax_bars.legend()
plt.tight_layout()
plt.show()
print("самая частая ошибка:", most_confused[0], "→", most_confused[1])
print("проверки пройдены")

Посмотрите на картинки, на которых сеть ошибается. Полносвязная сеть видит картинку
как 3072 независимых числа: она не знает, что соседние пиксели связаны, и опирается
на цвет и общий фон — небо у самолетов и кораблей, зелень у лягушек и оленей.

In [ ]:
network_answers = np.concatenate([final_network.forward(X_test[start:start + 5000]).argmax(axis=1)
                                  for start in range(0, len(y_test), 5000)])
mistakes = np.flatnonzero(network_answers != y_test)[:20]
show_images(cifar_X_test[mistakes].astype(np.float32) / 255, y_test[mistakes], network_answers[mistakes])

### Задача 7. Сравнение с библиотеками

Сравним сеть с двумя моделями sklearn в том же сетапе: те же стандартизованные `X_train`
и `X_test`. У моделей sklearn одинаковый интерфейс: конструктор с параметрами, `fit(X, y)` —
обучение, `predict(X)` — метки. Обучите, засеките время обучения (`time.perf_counter`)
и посчитайте accuracy на `X_test`:

| модель | переменная | класс и параметры |
|:--|:--|:--|
| логистическая регрессия | `logistic_model` | `LogisticRegression(max_iter=50)` |
| MLPClassifier | `sklearn_mlp` | `MLPClassifier(hidden_layer_sizes=(512, 256), solver="sgd", learning_rate_init=0.01, batch_size=128, max_iter=10, momentum=0.0, random_state=SEED)` |

У `MLPClassifier` та же архитектура, что у вашей сети. Таблица `comparison_table`: индекс —
`COMPARISON_ROWS`, столбцы — `COMPARISON_COLUMNS`; в строке своей сети — `final_test_accuracy`
и `final_train_time` из задачи 5.

In [ ]:
COMPARISON_ROWS = ["логистическая регрессия", "MLPClassifier", "своя сеть"]
COMPARISON_COLUMNS = ["accuracy на тесте", "время обучения, с"]

In [ ]:
logistic_model: LogisticRegression = ...     # обученная на X_train
sklearn_mlp: MLPClassifier = ...             # обученная на X_train
comparison_table: pd.DataFrame = ...         # индекс COMPARISON_ROWS, столбцы COMPARISON_COLUMNS

In [ ]:
# --- проверка ---
assert list(comparison_table.index) == COMPARISON_ROWS and list(comparison_table.columns) == COMPARISON_COLUMNS, "индекс и столбцы"
assert sklearn_mlp.hidden_layer_sizes == (512, 256), "у MLPClassifier архитектура как у своей сети"
assert np.isclose(comparison_table.loc["логистическая регрессия", "accuracy на тесте"], np.mean(logistic_model.predict(X_test) == y_test)), "accuracy логистической регрессии"
assert np.isclose(comparison_table.loc["своя сеть", "accuracy на тесте"], final_test_accuracy), "своя сеть — из задачи 5"
assert (comparison_table["время обучения, с"] > 0).all(), "время обучения положительное"
print("проверки пройдены")